# 04 — Feature Engineering
## Interview-Oriented ML Notes + Code

**Goal:** convert raw data into useful model inputs.

Topics covered:
- Missing values
- Missing indicators
- One-hot / ordinal encoding
- Scaling
- Outliers
- Log transforms
- Feature creation
- Interaction / polynomial features
- Date/time features
- Cyclical features
- Feature selection
- Correlation / multicollinearity
- Basic TF-IDF
- Data leakage
- ColumnTransformer + Pipeline
- Time-series features
- Interview workflow and cheat sheet


In [ ]:
# ============================================================
# SETUP
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, PolynomialFeatures
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

np.random.seed(42)


# 1. WHAT IS FEATURE ENGINEERING?

Raw data such as:

    age = 25
    salary = 60000
    city = "Chennai"

can become:

    age
    salary_log
    city_encoded
    salary_per_age
    signup_month

**Interview:** Feature engineering converts raw data into representations that help a model learn useful patterns.

Good features can make a simple model perform well; bad features can hurt even a powerful model.


In [ ]:
df = pd.DataFrame({
    "age": [22, 25, 35, 45, np.nan],
    "salary": [30000, 45000, 80000, 150000, 60000],
    "city": ["Chennai", "Mumbai", "Chennai", "Delhi", "Mumbai"],
    "experience": [1, 2, 8, 15, 5]
})
print(df)


# 2. MISSING VALUES

Numerical:
- mean
- median
- constant

Categorical:
- most_frequent
- constant such as `"Unknown"`

Median is often safer than mean when outliers exist.

**Important:** fit the imputer only on training data.


In [ ]:
imputer = SimpleImputer(strategy="median")
df[["age", "salary", "experience"]] = imputer.fit_transform(
    df[["age", "salary", "experience"]]
)
print(df)


# 3. MISSING-VALUE INDICATOR

Sometimes missingness itself is useful information.

`add_indicator=True` adds a flag showing which values were missing.


In [ ]:
x = pd.DataFrame({"salary": [30000, np.nan, 80000, np.nan, 60000]})

imputer = SimpleImputer(strategy="median", add_indicator=True)
result = imputer.fit_transform(x)

print(result)


# 4. CATEGORICAL ENCODING

### One-Hot Encoding
Use for categories with **no meaningful order**.

Example:
Chennai / Mumbai / Delhi

becomes separate binary columns.

### Ordinal Encoding
Use when order has meaning.

Example:
Low < Medium < High

Do not assign arbitrary numeric order to nominal categories.


In [ ]:
# One-hot
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(df[["city"]])

print(encoded)
print(encoder.get_feature_names_out(["city"]))

# Ordinal
levels = pd.DataFrame({"level": ["Low", "Medium", "High", "Medium"]})
ordinal = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
print(ordinal.fit_transform(levels))


# 5. FEATURE SCALING

### StandardScaler
    z = (x - mean) / std

### MinMaxScaler
    x' = (x - min) / (max - min)

### RobustScaler
Uses median and IQR, useful with strong outliers.

Scaling is important for:
- KNN
- K-Means
- SVM
- Logistic Regression
- Linear models in optimization
- PCA
- Neural networks

Usually unnecessary for:
- Decision Tree
- Random Forest
- Most tree-based models


In [ ]:
X = np.array([
    [20, 30000],
    [30, 50000],
    [40, 100000],
    [50, 200000]
])

print("StandardScaler:\n", StandardScaler().fit_transform(X))
print("MinMaxScaler:\n", MinMaxScaler().fit_transform(X))
print("RobustScaler:\n", RobustScaler().fit_transform(X))


# 6. OUTLIERS

An outlier may be:
- a data error
- a rare but valid event
- an important business case

Do **not** automatically delete it.

Possible approaches:
- investigate
- cap/winsorize
- log transform
- robust scaling
- remove only when justified


In [ ]:
x = pd.Series([10, 11, 12, 12, 13, 14, 15, 100])

q1 = x.quantile(.25)
q3 = x.quantile(.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Bounds:", lower, upper)
print("Outliers:", x[(x < lower) | (x > upper)].tolist())


# 7. LOG TRANSFORMATION

Useful for heavily right-skewed positive values.

Common:

    log1p(x) = log(1 + x)

`log1p` also handles zero safely.


In [ ]:
salary = np.array([30000, 40000, 50000, 60000, 1000000])
print(np.log1p(salary))


# 8. FEATURE CREATION

Use domain knowledge to create useful relationships.

Examples:

    total = price * quantity
    income_per_member = income / family_size
    bmi = weight / height**2

Feature creation is usually problem-specific.


In [ ]:
df = pd.DataFrame({
    "price": [100, 200, 300],
    "quantity": [2, 3, 5],
    "income": [50000, 70000, 100000],
    "family_size": [2, 4, 5]
})

df["total_amount"] = df["price"] * df["quantity"]
df["income_per_member"] = df["income"] / df["family_size"]

print(df)


# 9. INTERACTION / POLYNOMIAL FEATURES

Sometimes the relationship between features matters.

`PolynomialFeatures` can generate:
- original features
- powers
- interactions

Example:
`x1 * x2`

Be careful: many generated features can increase overfitting and computation.


In [ ]:
X = np.array([[2, 3], [4, 5], [6, 7]])

poly = PolynomialFeatures(degree=2, include_bias=False)
X_new = poly.fit_transform(X)

print(X_new)
print(poly.get_feature_names_out(["x1", "x2"]))


# 10. DATE/TIME FEATURES

From a timestamp extract:
- year
- month
- day
- day of week
- hour
- weekend
- quarter

For time series also consider:
- lag
- rolling mean/std
- difference
- seasonality

**Never use future information.**


In [ ]:
dates = pd.to_datetime([
    "2026-01-10 10:30",
    "2026-06-15 18:00",
    "2026-09-24 09:00"
])

d = pd.DataFrame({"date": dates})
d["year"] = d["date"].dt.year
d["month"] = d["date"].dt.month
d["day"] = d["date"].dt.day
d["day_of_week"] = d["date"].dt.dayofweek
d["hour"] = d["date"].dt.hour
d["is_weekend"] = (d["day_of_week"] >= 5).astype(int)

print(d)


# 11. CYCLICAL FEATURES

Hour 23 and hour 0 are adjacent in reality but far apart numerically.

Use:

    sin(2πx / period)
    cos(2πx / period)

Examples:
- hour → 24
- month → 12
- day of week → 7


In [ ]:
hours = np.array([0, 6, 12, 18, 23])

hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

print("sin:", hour_sin)
print("cos:", hour_cos)


# 12. FEATURE SELECTION

Feature engineering = creating useful features.

Feature selection = choosing useful features.

### Filter
Uses statistical properties:
- correlation
- ANOVA
- chi-square
- mutual information

### Wrapper
Uses model performance:
- RFE

### Embedded
Selection occurs during training:
- L1/Lasso
- tree importance

Why?
Too many irrelevant features can add noise, computation and overfitting.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)

print("Original:", X.shape)
print("Selected:", X_selected.shape)


In [ ]:
# Embedded selection using L1
model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    max_iter=2000
)

selector = SelectFromModel(model)
X_selected = selector.fit_transform(X, y)

print("Original:", X.shape[1])
print("Selected:", X_selected.shape[1])


# 13. CORRELATION

Pearson correlation:

    -1 <= r <= 1

+1 → strong positive linear relationship  
-1 → strong negative linear relationship  
0 → weak/no linear relationship

**Correlation is not causation.**

Low correlation does not automatically mean a feature is useless;
the relationship could be nonlinear.


In [ ]:
df_corr = pd.DataFrame({
    "x1": [1, 2, 3, 4, 5],
    "x2": [2, 4, 6, 8, 10],
    "x3": [5, 4, 3, 4, 5]
})

print(df_corr.corr())


# 14. MULTICOLLINEARITY

Multiple predictors contain highly overlapping information.

Important for linear/logistic regression because coefficients can become unstable.

Possible solutions:
- remove redundant features
- combine features
- L1/L2 regularization
- PCA when appropriate

Tree models are generally less sensitive for prediction, though correlated
features can still affect importance interpretation.


# 15. BASIC TEXT FEATURE ENGINEERING

Classical ML often uses:

### Bag of Words
Counts word occurrences.

### TF-IDF
Weights words based on their frequency in a document and rarity across documents.

    TF-IDF = TF × IDF

Useful for simple:
- text classification
- spam detection
- search

Modern NLP commonly uses embeddings/transformers.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [
    "machine learning is useful",
    "machine learning is powerful",
    "deep learning is powerful"
]

vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(texts)

print(vectorizer.get_feature_names_out())
print(X_text.shape)
print(X_text.toarray())


# 16. DATA LEAKAGE IN FEATURE ENGINEERING ⭐⭐⭐

Wrong:

    fit preprocessing on entire dataset
    ↓
    split train/test

The test data influenced preprocessing.

Correct:

    split
      ↓
    fit on train
      ↓
    transform train
      ↓
    transform test

This is one of the most important interview topics.


In [ ]:
# Correct pattern

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)


# 17. COLUMNTRANSFORMER

Real datasets contain mixed types.

Example:
- age → numerical
- salary → numerical
- city → categorical

ColumnTransformer applies different preprocessing to each type.


In [ ]:
data = pd.DataFrame({
    "age": [22, 30, 40, 50, 35, 28, 45, 32, 52, 26],
    "salary": [30000, 50000, 80000, 120000, 70000,
               40000, 100000, 60000, 140000, 35000],
    "city": ["Chennai", "Mumbai", "Delhi", "Chennai", "Mumbai",
             "Delhi", "Chennai", "Mumbai", "Delhi", "Chennai"],
    "target": [0, 0, 1, 1, 1, 0, 1, 0, 1, 0]
})

X = data.drop("target", axis=1)
y = data["target"]

numeric_features = ["age", "salary"]
categorical_features = ["city"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)

print("Processed shape:", X_processed.shape)


# 18. COMPLETE FEATURE ENGINEERING PIPELINE ⭐⭐⭐

Typical production pattern:

Raw data
→ train/test split
→ preprocessing
→ feature engineering
→ model
→ evaluation

Keeping transformations in a Pipeline makes the process reproducible and
prevents preprocessing leakage during cross-validation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))


# 19. TREE MODELS AND FEATURE ENGINEERING

Tree models usually do not require scaling.

They naturally learn threshold rules such as:

    salary < 70000 ?

But feature engineering can still help:
- domain-specific ratios
- date features
- aggregated features
- useful interactions

**Tree model ≠ no feature engineering.**


# 20. TIME-SERIES FEATURE ENGINEERING

Common features:

### Lag
    y(t-1), y(t-7), y(t-30)

### Rolling
    rolling_mean(7)
    rolling_std(7)

### Difference
    y(t) - y(t-1)

### Calendar
    hour, weekday, month, holiday

CRITICAL:
To predict time `t`, only use information available at or before `t`.

Never use:
    y(t+1)


In [ ]:
ts = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10),
    "demand": [10, 12, 15, 14, 18, 20, 19, 25, 23, 27]
})

ts["lag_1"] = ts["demand"].shift(1)
ts["lag_3"] = ts["demand"].shift(3)

# Shift first so today's value is not included in today's rolling feature
ts["rolling_mean_3"] = (
    ts["demand"].shift(1).rolling(3).mean()
)

print(ts)


# 21. INTERVIEW WORKFLOW

1. Understand target and prediction time.
2. Understand every feature.
3. Check data types.
4. Check missing values.
5. Check duplicates.
6. Check outliers and distributions.
7. Encode categorical variables.
8. Scale when required.
9. Create domain features.
10. Create date/time features.
11. Create lag/rolling features for time series.
12. Check leakage.
13. Check multicollinearity when relevant.
14. Select useful features.
15. Put preprocessing in Pipeline.
16. Validate with cross-validation.
17. Compare against a baseline.

**Most important rule:**
Every feature must be available when the prediction is made.


# 22. PARAMETERS CHEAT SHEET

## SimpleImputer
- `strategy`: mean / median / most_frequent / constant
- `fill_value`: replacement for constant strategy
- `add_indicator`: add missingness flags

## StandardScaler
- `with_mean`: center data
- `with_std`: scale to unit variance

## MinMaxScaler
- `feature_range`: output range

## RobustScaler
- `with_centering`
- `with_scaling`
Uses median/IQR.

## OneHotEncoder
- `handle_unknown="ignore"`: safely handles unseen categories
- `drop`: optionally remove a category
- `sparse_output`: sparse/dense output

## OrdinalEncoder
- `categories`: define ordering
- `handle_unknown`: behavior for unseen values

## PolynomialFeatures
- `degree`: maximum degree
- `include_bias`: add constant feature
- `interaction_only`: only interaction terms

## SelectKBest
- `score_func`: scoring function
- `k`: number of features

## ColumnTransformer
- transformer
- transformer name
- columns it applies to

## Pipeline
- ordered preprocessing/model steps


# 23. INTERVIEW CHEAT SHEET

**What is feature engineering?**  
Transforming raw data into useful model inputs.

**One-hot vs ordinal?**  
One-hot for unordered categories; ordinal when order has meaning.

**Why median imputation?**  
Less sensitive to outliers than mean.

**Why fit preprocessing only on train?**  
To prevent validation/test information from leaking into training.

**Why Pipeline?**  
Keeps transformations together and prevents leakage during CV.

**When is scaling important?**  
Distance-, margin-, and optimization-sensitive algorithms.

**Do trees require scaling?**  
Generally no.

**Feature engineering vs feature selection?**  
Engineering creates/transforms features; selection chooses a subset.

**What is target leakage?**  
A feature contains information that would not be available at prediction time.

**How do you handle outliers?**  
First determine whether they are errors or valid rare events; then choose an appropriate strategy.

**Most important rule?**  
Only use information that would genuinely be available when the prediction is made.


# FINAL MENTAL MODEL

Raw Data
   ↓
Missing values
   ↓
Categorical encoding
   ↓
Scaling / transformation
   ↓
Outlier handling
   ↓
Feature creation
   ↓
Date/time / text features
   ↓
Feature selection
   ↓
Leakage check
   ↓
Pipeline
   ↓
Model

### Remember these 5:
1. Feature engineering is domain-driven.
2. Preprocessing must not leak test/validation information.
3. Pipeline + ColumnTransformer are important practical tools.
4. Feature creation and feature selection are different.
5. Every feature must be available at prediction time.
